### Transformação de dados na Camada Silver

Leitura dos dados da camada Bronze, aplicação das regras de qualidade e integridade referencial, e armazenamento dos dados tratados na camada Silver.

1. Leitura das tabelas Delta da camada Bronze.
2. Leitura das tabelas de referência utilizadas nas validações.
3. Construção do catálogo unificado de SKUs a partir de `ecommerce_produtos` e `physical_produtos_pereciveis`.
4. Aplicação das regras de qualidade da tabela `food_avaliacoes_produto`.
5. Aplicação das regras de qualidade da tabela `food_estoque_lojas`.
6. Adição da coluna de auditoria `silver_processed_at`.
7. Criação das colunas `ano` e `mes` para particionamento.
8. Armazenamento dos dados no formato Delta na camada Silver, utilizando merge manual por chave primária e particionamento por ano e mês.

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE

import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, current_timestamp, year, month, trim, when, lit, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS

paths = {
    "food_avaliacoes_produto_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_avaliacoes_produto",
    "food_estoque_lojas_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_estoque_lojas",
    "food_avaliacoes_produto_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_avaliacoes_produto",
    "food_estoque_lojas_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_estoque_lojas",
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data"
}

In [0]:
# LEITURA DAS TABELAS BRONZE

df_bronze_avaliacoes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_bronze"])
)

df_bronze_estoque = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_bronze"])
)

In [0]:
# LEITURA DAS TABELAS DE REFERÊNCIA DA RAW

df_physical_lojas = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_lojas.csv")
)

df_ecommerce_produtos = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_produtos.csv")
)

df_physical_produtos = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_produtos_pereciveis.csv")
)

df_food_lotes = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/food_lotes_producao.csv")
)

df_ecommerce_pedidos = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_pedidos.csv")
)

df_physical_itens_venda_caixa = (
    spark.read.options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_itens_venda_caixa.csv")
)

In [0]:
# PREPARAÇÃO DAS TABELAS DE REFERÊNCIA

lojas_ref = df_physical_lojas.select("id_loja").distinct()

skus_ecommerce = df_ecommerce_produtos.select("sku").distinct()
skus_pereciveis = df_physical_produtos.select("sku").distinct()
skus_validos = skus_ecommerce.union(skus_pereciveis).distinct()

lotes_ref = df_food_lotes.select("id_lote").distinct()

pedidos_ref = df_ecommerce_pedidos.select("id_pedido").distinct()

In [0]:
# SILVER — FOOD_AVALIACOES_PRODUTO: DEDUPLICAÇÃO E TIPAGEM

df = df_bronze_avaliacoes

w = Window.partitionBy("id_avaliacao").orderBy(col("bronze_ingested_at").desc())

df = (
    df
    .filter(col("id_avaliacao").isNotNull())
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

df = df.drop("bronze_ingested_at", "bronze_source_file", "ano", "mes")

df = (
    df
    .withColumn("id_avaliacao", col("id_avaliacao").cast(IntegerType()))
    .withColumn("id_pedido", col("id_pedido").cast(IntegerType()))
    .withColumn("id_cliente", col("id_cliente").cast(IntegerType()))
    .withColumn("sku", trim(col("sku")))
    .withColumn("nota", col("nota").cast(IntegerType()))
    .withColumn("dt_avaliacao", col("dt_avaliacao").cast(TimestampType()))
    .withColumn("verificada", col("verificada").cast(BooleanType()))
)

df_silver_avaliacoes = df

In [0]:
# SILVER — FOOD_AVALIACOES_PRODUTO: FLAGS DE QUALIDADE

df = df_silver_avaliacoes

df = df.withColumn(
    "_flag_nota_invalida",
    when(
        col("nota").isNull() | ~col("nota").between(1, 5),
        lit(True)
    ).otherwise(lit(False))
)

pedidos_ref_flag = pedidos_ref.select(col("id_pedido").alias("id_pedido_ref"))

df = (
    df.join(
        pedidos_ref_flag,
        df["id_pedido"] == col("id_pedido_ref"),
        "left"
    )
    .withColumn(
        "_flag_pedido_verificada_invalido",
        when(
            (col("verificada") == True) &
            (col("id_pedido").isNull() | col("id_pedido_ref").isNull()),
            lit(True)
        ).otherwise(lit(False))
    )
    .drop("id_pedido_ref")
)

df = df.withColumn(
    "_flag_pedido_nao_verificada_invalido",
    when(
        (col("verificada") == False) & col("id_pedido").isNotNull(),
        lit(True)
    ).otherwise(lit(False))
)

skus_ref_flag = skus_validos.select(col("sku").alias("sku_ref"))

df = (
    df.join(
        skus_ref_flag,
        df["sku"] == col("sku_ref"),
        "left"
    )
    .withColumn(
        "_flag_sku_sem_catalogo",
        when(col("sku_ref").isNull(), lit(True)).otherwise(lit(False))
    )
    .drop("sku_ref")
)

df_silver_avaliacoes = df

In [0]:
# SILVER — FOOD_AVALIACOES_PRODUTO: AUDITORIA E PARTIÇÃO

df = df_silver_avaliacoes

df = (
    df
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("ano", year(col("dt_avaliacao")))
    .withColumn("mes", month(col("dt_avaliacao")))
)

df_silver_avaliacoes = df

In [0]:
# SILVER — FOOD_ESTOQUE_LOJAS: DEDUPLICAÇÃO E TIPAGEM

df = df_bronze_estoque

w = Window.partitionBy(
    "id_loja", "sku", "id_lote", "dt_snapshot"
).orderBy(col("bronze_ingested_at").desc())

df = (
    df
    .filter(
        col("id_loja").isNotNull() &
        col("sku").isNotNull() &
        col("dt_snapshot").isNotNull()
    )
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

df = df.drop("bronze_ingested_at", "bronze_source_file", "ano", "mes")

df = (
    df
    .withColumn("id_loja", col("id_loja").cast(IntegerType()))
    .withColumn("sku", trim(col("sku")))
    .withColumn("quantidade_disponivel", col("quantidade_disponivel").cast(IntegerType()))
    .withColumn("estoque_minimo", col("estoque_minimo").cast(IntegerType()))
    .withColumn("id_lote", col("id_lote").cast(IntegerType()))
    .withColumn("dt_snapshot", col("dt_snapshot").cast(TimestampType()))
    .withColumn("dt_fabricacao", col("dt_fabricacao").cast(TimestampType()))
    .withColumn("dt_validade", col("dt_validade").cast(TimestampType()))
)

df_silver_estoque = df

In [0]:
# SILVER — FOOD_ESTOQUE_LOJAS: FLAGS DE QUALIDADE

df = df_silver_estoque

lojas_ref_flag = lojas_ref.select(col("id_loja").alias("id_loja_ref"))

df = (
    df.join(lojas_ref_flag, df["id_loja"] == col("id_loja_ref"), "left")
    .withColumn(
        "_flag_loja_sem_cadastro",
        when(col("id_loja_ref").isNull(), lit(True)).otherwise(lit(False))
    )
    .drop("id_loja_ref")
)

skus_ref_flag = skus_validos.select(col("sku").alias("sku_ref"))

df = (
    df.join(skus_ref_flag, df["sku"] == col("sku_ref"), "left")
    .withColumn(
        "_flag_sku_sem_catalogo",
        when(col("sku_ref").isNull(), lit(True)).otherwise(lit(False))
    )
    .drop("sku_ref")
)

df = df.withColumn(
    "_flag_quantidade_invalida",
    when(
        col("quantidade_disponivel").isNull() |
        (col("quantidade_disponivel") < 0),
        lit(True)
    ).otherwise(lit(False))
)

lotes_ref_flag = lotes_ref.select(col("id_lote").alias("id_lote_ref"))

df = (
    df.join(lotes_ref_flag, df["id_lote"] == col("id_lote_ref"), "left")
    .withColumn(
        "_flag_lote_sem_cadastro",
        when(
            col("id_lote").isNotNull() & col("id_lote_ref").isNull(),
            lit(True)
        ).otherwise(lit(False))
    )
    .drop("id_lote_ref")
)

df = df.withColumn(
    "_flag_snapshot_fora_validade",
    when(
        col("dt_snapshot").isNull() |
        col("dt_fabricacao").isNull() |
        col("dt_validade").isNull() |
        (col("dt_snapshot") < col("dt_fabricacao")) |
        (col("dt_snapshot") > col("dt_validade")),
        lit(True)
    ).otherwise(lit(False))
)

df_silver_estoque = df

In [0]:
# SILVER — FOOD_ESTOQUE_LOJAS: ENRIQUECIMENTO COM ITENS DE VENDA EM CAIXA

df_caixa_ref = df_physical_itens_venda_caixa.dropDuplicates(["id_loja", "sku"])

colunas_caixa = [
    col(c).alias(f"caixa_{c}")
    for c in df_caixa_ref.columns
    if c not in ["id_loja", "sku"]
]

df_silver_estoque = (
    df_silver_estoque
    .join(
        df_caixa_ref.select(
            col("id_loja").alias("caixa_id_loja"),
            col("sku").alias("caixa_sku"),
            *colunas_caixa
        ),
        (df_silver_estoque["id_loja"] == col("caixa_id_loja")) &
        (df_silver_estoque["sku"] == col("caixa_sku")),
        "left"
    )
    .drop("caixa_id_loja", "caixa_sku")
)

In [0]:
# SILVER — FOOD_ESTOQUE_LOJAS: AUDITORIA E PARTIÇÃO

df = df_silver_estoque

df = (
    df
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("ano", year(col("dt_snapshot")))
    .withColumn("mes", month(col("dt_snapshot")))
)

df_silver_estoque = df

In [0]:
# FUNÇÃO DE MERGE MANUAL NA SILVER

def save_silver_merge(df_new, silver_path, pk_cols):
    try:
        spark.read.format("binaryFile").options(**adls_options).load(silver_path).limit(1).count()
        silver_exists = True
    except Exception:
        silver_exists = False

    if silver_exists:
        df_existing = (
            spark.read.format("delta")
            .options(**adls_options)
            .load(silver_path)
        )
        df_merged = df_existing.unionByName(df_new, allowMissingColumns=True)
    else:
        df_merged = df_new

    w = Window.partitionBy(*pk_cols).orderBy(col("silver_processed_at").desc())

    df_final = (
        df_merged
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("ano", "mes")
        .options(**adls_options)
        .save(silver_path)
    )

    return df_final

In [0]:
# GRAVAÇÃO DAS TABELAS NA SILVER

df_silver_avaliacoes_final = save_silver_merge(
    df_new=df_silver_avaliacoes,
    silver_path=paths["food_avaliacoes_produto_silver"],
    pk_cols=["id_avaliacao"]
)

df_silver_estoque_final = save_silver_merge(
    df_new=df_silver_estoque,
    silver_path=paths["food_estoque_lojas_silver"],
    pk_cols=["id_loja", "sku", "id_lote", "dt_snapshot"]
)